## Task 8: Fine-Tune Cellpose on 35 Training FOVs

Convert GT boundary polygons to integer masks, then fine-tune Cellpose `nuclei` on FOVs 001–035 and evaluate on held-out FOVs 036–040.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.io import load_fov_images
from src.train_cellpose import boundaries_to_mask

DATA_ROOT = "/scratch/pl2820/competition"
# fov_metadata.csv lives at the competition root (not reference/)
meta  = pd.read_csv(f"{DATA_ROOT}/fov_metadata.csv").set_index("fov")
cells = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/cell_boundaries_train.csv", index_col=0)


### Step 1: Verify GT mask on FOV_001

In [ ]:
fov_x = meta.loc["FOV_001", "fov_x"]
fov_y = meta.loc["FOV_001", "fov_y"]
dapi, polyt = load_fov_images(f"{DATA_ROOT}/train/FOV_001")
mask = boundaries_to_mask(cells, "FOV_001", fov_x, fov_y)

gt_count = len(cells[cells.index.str.startswith("FOV_001")])
print(f"GT cells in CSV : {gt_count}")
print(f"Cells in mask   : {mask.max()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(dapi[2], cmap="gray", vmin=0, vmax=np.percentile(dapi[2], 99))
axes[0].set_title("DAPI z2 — FOV_001")
axes[1].imshow(mask > 0, cmap="gray")
axes[1].set_title("GT mask (white = any cell)")
plt.tight_layout()
plt.savefig("mask_check_fov001.png", dpi=80)
plt.show()
print("Saved mask_check_fov001.png")


### Step 2: Build training set (FOVs 001–035) and fine-tune

In [ ]:
from cellpose import models as cp_models, train as cp_train

train_images = []
train_masks  = []
train_fovs   = [f"FOV_{i:03d}" for i in range(1, 36)]

for fov_name in train_fovs:
    fov_dir = f"{DATA_ROOT}/train/{fov_name}"
    if not os.path.exists(fov_dir):
        print(f"  Missing: {fov_name}")
        continue
    try:
        dapi, polyt = load_fov_images(fov_dir)
        fov_x = meta.loc[fov_name, "fov_x"]
        fov_y = meta.loc[fov_name, "fov_y"]
        m = boundaries_to_mask(cells, fov_name, fov_x, fov_y)
        if m.max() == 0:
            print(f"  No cells in mask: {fov_name}")
            continue
        train_images.append(np.stack([polyt[2], dapi[2]], axis=0))
        train_masks.append(m)
        print(f"  Loaded {fov_name}: {m.max()} cells")
    except Exception as exc:
        print(f"  Skipping {fov_name}: {exc}")

print(f"\nTraining on {len(train_images)} FOVs")

os.makedirs("models", exist_ok=True)
# Fine-tune starting from the nuclei pretrained model (matches colab baseline)
base_model = cp_models.CellposeModel(gpu=True, model_type="cyto2")
model_path = cp_train.train_seg(
    base_model.net,
    train_data=train_images,
    train_labels=train_masks,
    channels=[1, 2],
    save_path="models/",
    n_epochs=100,
    learning_rate=0.005,
    weight_decay=1e-5,
    batch_size=8,
    model_name="cellpose_finetuned",
)
print(f"\nSaved fine-tuned model: {model_path}")


### Step 3: Evaluate fine-tuned model on validation FOVs (036–040)

Uses direct pixel lookup: `cluster_id = mask[pixel_y, pixel_x]` — consistent with submission format (integer 0 = background).

In [ ]:
from sklearn.metrics import adjusted_rand_score

PIXEL_SIZE = 0.109

finetuned = cp_models.CellposeModel(gpu=True, pretrained_model="models/cellpose_finetuned")
val_fovs = [f"FOV_{i:03d}" for i in range(36, 41)]
spots_train = pd.read_csv(f"{DATA_ROOT}/train/ground_truth/spots_train.csv")

ari_scores = {}
for fov_name in val_fovs:
    fov_dir = f"{DATA_ROOT}/train/{fov_name}"
    if not os.path.exists(fov_dir):
        print(f"  Missing: {fov_name}")
        continue

    dapi, polyt = load_fov_images(fov_dir)
    fov_x = meta.loc[fov_name, "fov_x"]
    fov_y = meta.loc[fov_name, "fov_y"]

    pred_masks, _, _ = finetuned.eval(dapi[2], diameter=30, channels=[1, 2])
    gt_mask = boundaries_to_mask(cells, fov_name, fov_x, fov_y)

    fov_spots = spots_train[spots_train["fov"] == fov_name].copy()
    # Pixel lookup: convert global µm -> pixel, then read mask value
    px = np.clip(((fov_spots["global_x"].values - fov_x) / PIXEL_SIZE).astype(int), 0, 2047)
    py = np.clip(((fov_spots["global_y"].values - fov_y) / PIXEL_SIZE).astype(int), 0, 2047)
    pred_ids = pred_masks[py, px]
    gt_ids   = gt_mask[py, px]

    ari = adjusted_rand_score(gt_ids, pred_ids)
    ari_scores[fov_name] = ari
    print(f"{fov_name}: ARI = {ari:.4f}  ({pred_masks.max()} cells detected)")

mean_ari = np.mean(list(ari_scores.values())) if ari_scores else 0.0
print(f"\nMean validation ARI : {mean_ari:.4f}")
print(f"Baseline (pretrained): 0.632")
print(f"Improvement          : {mean_ari - 0.632:+.4f}")
